# TestSphere.AI Experiment Notebook
This notebook loads the dataset, dependency map, test coverage, and failure history to evaluate the change-impact test selection.
It runs a baseline (all tests) and TestSphere.AI's smart selection, then compares execution time reduction and affected-test false negatives.

In [ ]:
import sys
import os
from pathlib import Path
sys.path.insert(0, str(Path(os.getcwd()).parent))

from Engine.database import get_connection
from Engine.test_selector import run_selection
from Simulation.baseline_runner import run_baseline
from Simulation.smart_runner import run_smart
from Engine.metrics_calculator import calculate_experiment_metrics, calculate_time_metrics
import pandas as pd

print("Environment setup complete.")

In [ ]:
# 1-4. Load Data
def load_data(table_name):
    conn = get_connection()
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    return df

coverage_df = load_data("test_coverage")
deps_df = load_data("dependency_map")
failures_df = load_data("failure_history")
changes_df = load_data("code_changes")

print(f"Loaded {len(coverage_df)} tests, {len(deps_df)} dependencies, {len(failures_df)} failures.")

In [ ]:
# 5. Define code-change scenarios
scenarios = [
    {"name": "Scenario 1: Payment logic change", "files": ["payment/payment.py"], "module": "Payment"},
    {"name": "Scenario 2: Auth login change", "files": ["auth/login.py"], "module": "Authentication"},
    {"name": "Scenario 3: Transfer change", "files": ["transaction/txn.py"], "module": "Transaction"},
    {"name": "Scenario 4: Notification change", "files": ["notification/push.py"], "module": "Notification"},
    {"name": "Scenario 5: Multi-file change", "files": ["payment/payment.py", "auth/login.py"], "module": "Payment"}
]

In [ ]:
# 6-11. Run Baseline vs Target and compute metrics
results = []
for s in scenarios:
    print(f"\nRunning {s['name']}...")
    
    # 6. Baseline
    baseline = run_baseline()
    
    # 7. TestSphere.AI selection
    selection = run_selection(changed_files=s["files"], change_type="MODIFIED", module=s["module"], is_security_sensitive=False, risk_level="MEDIUM")
    decisions = selection["decisions"]
    
    # 8. Compare selected tests
    smart_exec = run_smart(decisions)
    
    # 9. Calculate runtime reduction
    all_tests = coverage_df.to_dict('records')
    selected_tests = [t for t in all_tests if any(d["test_id"] == t["test_id"] and d["decision"] == "RUN" for d in decisions)]
    time_metrics = calculate_time_metrics(all_tests, selected_tests)
    
    # 10. Calculate false negatives
    metrics = calculate_experiment_metrics(decisions, change_id="CHG001" if s["files"] == ["payment/payment.py"] else "CHG001")
    
    res = {
        "Scenario": s['name'],
        "Total Tests": baseline['total_tests'],
        "Executed Tests (Baseline)": baseline['executed'],
        "Executed Tests (Target)": smart_exec['executed'],
        "Skipped Tests (Target)": smart_exec['skipped'],
        "Time Reduction %": time_metrics['time_reduction_pct'],
        "False Negatives": metrics['false_negatives']
    }
    results.append(res)

results_df = pd.DataFrame(results)
display(results_df)